In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Optional, Sequence

from trace_comparison.triplet_accuracy import triplet_accuracy, relatedness_levels
from trace_comparison.data.dataset import load_trace_dataset
from trace_comparison.blocks.implementations.same_shape import SubtractPuzzle, L2Norm
from trace_comparison.blocks.implementations.trace_emb import PrincipalComponentAnalysis, UniformManifoldApproximation, ClassificationLatent
from trace_comparison.blocks.implementations.alignment import Resample 
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()
base_path = Path(os.path.join(os.getcwd()))

In [ ]:
dataset = load_trace_dataset(base_path / "data" / "traces_dataset" / "nonograms" / "base_1000.jsonl")
resample_block = Resample(k=5)
subtract_puzzle_block = SubtractPuzzle()
l2_norm_block = L2Norm()
pca_block = PrincipalComponentAnalysis(emb_dim=50, whiten=False)
umap_block = UniformManifoldApproximation(emb_dim=2, n_neighbors=200, min_dist=0.1)
latent_checkpoint = base_path / "checkpoints"
latent_block = ClassificationLatent(checkpoint_dir=latent_checkpoint, device="cpu", model_version="10")

In [ ]:
def class_colors(n: int, cmap: str = "turbo") -> np.ndarray:
    """One distinct RGBA colour per class (no cycling). Returns shape (n, 4)."""
    return plt.get_cmap(cmap)(np.linspace(0.05, 0.95, n))


def density_style(n_points: int) -> tuple[float, float]:
    """Point size and alpha that scale down as the number of points grows."""
    size = float(np.clip(4000 / max(n_points, 1), 2, 8))
    alpha = float(np.clip(50 / np.sqrt(max(n_points, 1)), 0.15, 0.6))
    return size, alpha


def _padded_limits(X2: np.ndarray, pad: float = 0.04):
    """Axis limits covering all points with a small margin."""
    lo, hi = X2.min(axis=0), X2.max(axis=0)
    margin = (hi - lo) * pad
    return (lo[0] - margin[0], hi[0] + margin[0]), (lo[1] - margin[1], hi[1] + margin[1])


def _clean_axes(ax, xlim=None, ylim=None, equal: bool = False):
    """Strip ticks/spines so the embedding (whose units are arbitrary) reads cleanly."""
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if equal:
        ax.set_aspect("equal", adjustable="box")



In [ ]:

# --------------------------------------------------------------------------
# overview: every class on one axis
# --------------------------------------------------------------------------
def plot_overview(
    X2: np.ndarray,
    y: np.ndarray,
    names: Sequence[str],
    colors: Optional[np.ndarray] = None,
    title: str = "UMAP of embeddings",
    annotate_medians: bool = True,
    figsize: tuple[float, float] = (10, 8),
    legend_cols: Optional[int] = None,
):
    """Scatter of all points coloured by class, with a robust median marker per class.

    Legend entries include class sizes. If `annotate_medians` is True, each median
    is labelled with its class index so clusters can be identified without
    cross-referencing the legend colours.
    """
    n_labels = len(names)
    colors = class_colors(n_labels) if colors is None else colors
    size, alpha = density_style(len(X2))
    xlim, ylim = _padded_limits(X2)

    fig, ax = plt.subplots(figsize=figsize)
    for i in range(n_labels):
        pts = X2[y == i]
        if len(pts) == 0:
            continue
        ax.scatter(pts[:, 0], pts[:, 1], s=size, alpha=alpha, color=colors[i],
                   linewidths=0, rasterized=True, label=f"{i}: {names[i]} (n={len(pts)})")
        mx, my = np.median(pts, axis=0)          # median: robust to outliers
        ax.scatter(mx, my, s=110, color=colors[i], edgecolor="k",
                   linewidths=1.0, marker="X", zorder=5)
        if annotate_medians:
            ax.annotate(str(i), (mx, my), xytext=(4, 4), textcoords="offset points",
                        fontsize=8, fontweight="bold", zorder=6,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))

    _clean_axes(ax, xlim, ylim, equal=True)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(title, fontsize=13)

    ncol = legend_cols or max(1, int(np.ceil(n_labels / 25)))
    leg = ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8,
                    ncol=ncol, frameon=False, markerscale=max(1.0, 6 / size),
                    title="class (n)", title_fontsize=9)
    for handle in leg.legend_handles:            # legend markers fully opaque
        handle.set_alpha(1)

    fig.tight_layout()
    return fig, ax


# --------------------------------------------------------------------------
# small multiples: one panel per class
# --------------------------------------------------------------------------
def plot_small_multiples(
    X2: np.ndarray,
    y: np.ndarray,
    names: Sequence[str],
    colors: Optional[np.ndarray] = None,
    ncols: int = 6,
    panel_size: float = 2.4,
    title: str = "Per-class view (grey = all other points)",
    order: Optional[Sequence[int]] = None,
):
    """Grid with one panel per class, highlighted over a grey backdrop of all points.

    `order` optionally reorders the panels (e.g. by class size or by median position).
    """
    n_labels = len(names)
    colors = class_colors(n_labels) if colors is None else colors
    order = list(range(n_labels)) if order is None else list(order)
    nrows = int(np.ceil(len(order) / ncols))
    xlim, ylim = _padded_limits(X2)

    fig, axes = plt.subplots(nrows, ncols, sharex=True, sharey=True, squeeze=False,
                             figsize=(panel_size * ncols, panel_size * nrows * 1.05),
                             constrained_layout=True)
    flat = axes.ravel()

    for ax, i in zip(flat, order):
        ax.scatter(X2[:, 0], X2[:, 1], s=1, c="lightgrey", alpha=0.4,
                   linewidths=0, rasterized=True)
        pts = X2[y == i]
        ax.scatter(pts[:, 0], pts[:, 1], s=3, color=colors[i], alpha=0.85,
                   linewidths=0, rasterized=True)
        ax.set_title(f"{names[i]} (n={len(pts)})", fontsize=9)
        _clean_axes(ax, xlim, ylim)

    for ax in flat[len(order):]:                 # hide unused panels
        ax.axis("off")

    fig.suptitle(title, fontsize=13)
    return fig, axes


In [ ]:
x = dataset.traces
for block in [resample_block, subtract_puzzle_block, l2_norm_block, pca_block, umap_block]:
    x = block(x)
n_labels, n_samples, n_features = x.shape
x = x.reshape(-1, n_features)
y = np.repeat(np.arange(n_labels), n_samples)
colours = class_colors(n_labels)
plot_overview(x, y, dataset.solvers, colours)
plot_small_multiples(x, y, dataset.solvers, colours)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X2_ = x.reshape(n_labels, n_samples, 2)
def split(i):
    return KMeans(2, n_init=10, random_state=0).fit_predict(X2_[i])

names = list(dataset.solvers)
a, b = names.index("ga_rlt_1"), names.index("ga_rlt_2")
print(adjusted_rand_score(split(a), split(b)))   # near 1 => puzzle-driven
a, b = names.index("ga_rlt_1"), names.index("ga_tmt_1")
print(adjusted_rand_score(split(a), split(b)))   # near 1 => puzzle-driven
a, b = names.index("ga_rlt_1"), names.index("nn_rml_1")
print(adjusted_rand_score(split(a), split(b)))   # near 1 => puzzle-driven

In [ ]:
import pandas as pd
ca, cb = split(a), split(b)
print(pd.crosstab(ca, cb))

In [ ]:
x = dataset.traces
for block in [resample_block, latent_block, subtract_puzzle_block, l2_norm_block, pca_block]:
    x = block(x)
n_labels, n_samples, n_features = x.shape
x = x.reshape(-1, n_features)
y = np.repeat(np.arange(n_labels), n_samples)
print(x.shape, y.shape)
colours = class_colors(n_labels)
plot_overview(x, y, dataset.solvers, colours)
plot_small_multiples(x, y, dataset.solvers, colours)
plt.show()
Xp = x.reshape(n_labels, n_samples, -1)[a]          # PCA output, before UMAP
c = split(a)
d_within = np.mean([np.linalg.norm(Xp[c==k] - Xp[c==k].mean(0), axis=1).mean() for k in (0,1)])
d_between = np.linalg.norm(Xp[c==0].mean(0) - Xp[c==1].mean(0))
print(d_within, d_between)